In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from datetime import datetime
import time
from typing import List, Dict, Any
import gc

Kaggle data implementation

In [ ]:
# Try to import GPU libraries, fallback to CPU
try:
    import cudf
    import cupy as cp
    from cuml.preprocessing import LabelEncoder as cuLabelEncoder
    GPU_AVAILABLE = True
    print("GPU libraries found. Using GPU acceleration.")
except ImportError:
    GPU_AVAILABLE = False
    print("GPU libraries not found. Using CPU implementation.")

# Import CPU libraries for fallback
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

class GPUOptimizedRetailAnalytics:
    def __init__(self):
        self.contextual_rules = {}
        self.products_df = None
        self.transaction_contexts = []
        self.analytics_data = {}
        self.optimization_settings = {
            'max_transactions': 100000,
            'sampling_enabled': True,
            'sample_size': 20000,
            'min_support': 0.01,
            'min_confidence': 0.3,
            'max_len': 2,
            'chunk_size': 50000
        }

    def set_optimization_settings(self, **settings):
        """Configure optimization parameters"""
        self.optimization_settings.update(settings)
        print("Optimization settings updated:", self.optimization_settings)

    def load_instacart_data_optimized(self, data_paths):
        """
        Load Instacart dataset with memory optimization
        """
        print("Loading Instacart datasets with optimization...")

        # Load datasets with optimized data types
        aisles = self._load_csv_optimized(f"{data_paths}/aisles.csv")
        order_products_prior = self._load_csv_optimized(f"{data_paths}/order_products__prior.csv")
        order_products_train = self._load_csv_optimized(f"{data_paths}/order_products__train.csv")
        orders = self._load_csv_optimized(f"{data_paths}/orders.csv")
        products = self._load_csv_optimized(f"{data_paths}/products.csv")

        # Clean datasets
        aisles = self._clean_dataframe_optimized(aisles, 'aisles')
        order_products_prior = self._clean_dataframe_optimized(order_products_prior, 'order_products_prior')
        order_products_train = self._clean_dataframe_optimized(order_products_train, 'order_products_train')
        orders = self._clean_dataframe_optimized(orders, 'orders')
        products = self._clean_dataframe_optimized(products, 'products')

        # Sample data if too large
        if len(order_products_prior) > self.optimization_settings['max_transactions']:
            sample_size = self.optimization_settings['sample_size']
            print(f"Sampling {sample_size} records from order_products_prior...")
            order_products_prior = order_products_prior.sample(n=sample_size, random_state=42)

        # Combine order products
        print("Combining order products...")
        all_order_products = pd.concat([order_products_prior, order_products_train], ignore_index=True)

        # Load departments
        try:
            departments = self._load_csv_optimized(f"{data_paths}/departments.csv")
        except:
            print("Departments file not found, creating placeholder...")
            departments = pd.DataFrame({'department_id': [], 'department': []})

        # Create product catalog (optimized merge)
        print("Creating product catalog...")
        products_full = products.merge(aisles, on='aisle_id', how='left')
        if not departments.empty:
            products_full = products_full.merge(departments, on='department_id', how='left')
        else:
            products_full['department'] = 'unknown'

        # Optimized merging - process in chunks if needed
        print("Merging orders with products...")
        if len(orders) > 100000:
            self.sales_data = self._chunked_merge(orders, all_order_products, products_full)
        else:
            # Merge orders with order products
            orders_with_products = orders.merge(all_order_products, on='order_id')
            # Final merge with product information
            self.sales_data = orders_with_products.merge(products_full, on='product_id')

        # Store reference datasets
        self.products_df = products_full
        self.orders_df = orders
        self.order_products_df = all_order_products
        self.aisles_df = aisles

        print(f"Loaded {len(self.sales_data):,} sales records")
        print(f"Unique products: {self.sales_data['product_id'].nunique():,}")
        print(f"Unique orders: {self.sales_data['order_id'].nunique():,}")
        print(f"Unique customers: {self.sales_data['user_id'].nunique():,}")

        # Clear memory
        del aisles, order_products_prior, order_products_train, orders, products, departments
        gc.collect()

        return self.sales_data

    def _load_csv_optimized(self, file_path):
        """Load CSV with optimized data types and GPU support"""
        if GPU_AVAILABLE:
            try:
                return cudf.read_csv(file_path)
            except:
                print(f"GPU loading failed for {file_path}, falling back to CPU")
                return pd.read_csv(file_path)
        else:
            return pd.read_csv(file_path)

    def _chunked_merge(self, df1, df2, df3):
        """Merge large datasets in chunks to save memory"""
        print("Performing chunked merge for large datasets...")
        chunk_size = self.optimization_settings['chunk_size']
        chunks = []

        for i in range(0, len(df1), chunk_size):
            print(f"Processing chunk {i//chunk_size + 1}...")
            chunk = df1.iloc[i:i+chunk_size].copy()

            # Merge with second dataframe
            chunk_merged = chunk.merge(df2, on='order_id')

            # Merge with third dataframe
            chunk_final = chunk_merged.merge(df3, on='product_id')

            chunks.append(chunk_final)

            # Clear memory
            del chunk, chunk_merged, chunk_final
            gc.collect()

        result = pd.concat(chunks, ignore_index=True)
        return result

    def _clean_dataframe_optimized(self, df, df_name):
        """Optimized cleaning with memory efficiency"""
        print(f"Cleaning {df_name}...")

        # Convert to pandas if it's a GPU dataframe
        if GPU_AVAILABLE and hasattr(df, 'to_pandas'):
            df = df.to_pandas()

        # Optimize data types
        for col in df.columns:
            if df[col].dtype == 'object':
                # Convert to category if low cardinality
                if df[col].nunique() / len(df) < 0.5:
                    df[col] = df[col].astype('category')
            elif df[col].dtype in ['int64', 'float64']:
                # Downcast numeric columns
                df[col] = pd.to_numeric(df[col], downcast='integer')

        # Handle missing values efficiently
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].isnull().sum() > 0:
                df[col] = df[col].fillna(0)

        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        for col in categorical_cols:
            if df[col].isnull().sum() > 0:
                df[col] = df[col].fillna('unknown')

        return df

    def prepare_transactions_memory_efficient(self):
        """
        Prepare transactions with memory-efficient processing
        """
        print("Preparing transactions with memory optimization...")

        if self.optimization_settings['sampling_enabled'] and len(self.sales_data) > self.optimization_settings['sample_size']:
            sample_size = min(self.optimization_settings['sample_size'], len(self.sales_data))
            print(f"Sampling {sample_size} transactions...")
            sales_sample = self.sales_data.sample(n=sample_size, random_state=42)
        else:
            sales_sample = self.sales_data

        # Process in chunks for very large datasets
        if len(sales_sample) > 50000:
            return self._prepare_transactions_chunked(sales_sample)

        # Standard processing for smaller datasets
        return self._prepare_transactions_standard(sales_sample)

    def _prepare_transactions_standard(self, sales_df):
        """Standard transaction preparation"""
        # Calculate order-level metrics efficiently
        order_metrics = sales_df.groupby('order_id').agg({
            'user_id': 'first',
            'order_dow': 'first',
            'order_hour_of_day': 'first',
            'days_since_prior_order': 'first',
            'product_id': 'count'
        }).reset_index()

        order_metrics.rename(columns={'product_id': 'total_items'}, inplace=True)

        # Get product lists
        order_products = sales_df.groupby('order_id')['product_id'].apply(list).reset_index()

        # Merge efficiently
        transactions_with_context = order_metrics.merge(order_products, on='order_id')

        # Create transaction contexts
        self.transaction_contexts = []
        for _, row in transactions_with_context.iterrows():
            if isinstance(row['product_id'], list) and len(row['product_id']) > 0:
                self.transaction_contexts.append({
                    'order_id': row['order_id'],
                    'products': row['product_id'],
                    'total_items': row['total_items'],
                    'day_of_week': self._get_day_name(row['order_dow']),
                    'time_of_day': self._categorize_time_of_day(row['order_hour_of_day']),
                    'basket_size': self._categorize_basket_size(row['total_items'])
                })

        print(f"Prepared {len(self.transaction_contexts)} transactions")
        return self.transaction_contexts

    def _prepare_transactions_chunked(self, sales_df):
        """Prepare transactions in chunks for very large datasets"""
        print("Using chunked transaction preparation...")
        chunk_size = 10000
        self.transaction_contexts = []

        unique_orders = sales_df['order_id'].unique()

        for i in range(0, len(unique_orders), chunk_size):
            print(f"Processing order chunk {i//chunk_size + 1}...")
            order_chunk = unique_orders[i:i+chunk_size]
            chunk_data = sales_df[sales_df['order_id'].isin(order_chunk)]

            # Process chunk
            order_products = chunk_data.groupby('order_id')['product_id'].apply(list).reset_index()
            order_metrics = chunk_data.groupby('order_id').agg({
                'order_dow': 'first',
                'order_hour_of_day': 'first',
                'product_id': 'count'
            }).reset_index()

            order_metrics.rename(columns={'product_id': 'total_items'}, inplace=True)

            # Merge and create contexts
            chunk_contexts = order_metrics.merge(order_products, on='order_id')
            for _, row in chunk_contexts.iterrows():
                if isinstance(row['product_id'], list) and len(row['product_id']) > 0:
                    self.transaction_contexts.append({
                        'order_id': row['order_id'],
                        'products': row['product_id'],
                        'total_items': row['total_items'],
                        'day_of_week': self._get_day_name(row['order_dow']),
                        'time_of_day': self._categorize_time_of_day(row['order_hour_of_day']),
                        'basket_size': self._categorize_basket_size(row['total_items'])
                    })

            # Clear memory
            del order_chunk, chunk_data, order_products, order_metrics, chunk_contexts
            gc.collect()

        print(f"Prepared {len(self.transaction_contexts)} transactions")
        return self.transaction_contexts

    def generate_rules_optimized(self):
        """
        Generate association rules with performance optimizations
        """
        print("Generating optimized association rules...")

        if len(self.transaction_contexts) < 10:
            print("Insufficient transactions for rule generation")
            return

        # Use adaptive thresholds
        data_size = len(self.transaction_contexts)
        if data_size < 1000:
            min_support = 0.02
            min_confidence = 0.4
            max_len = 2
        elif data_size < 10000:
            min_support = 0.01
            min_confidence = 0.35
            max_len = 2
        else:
            min_support = 0.005
            min_confidence = 0.3
            max_len = 3

        print(f"Using thresholds: support={min_support}, confidence={min_confidence}, max_len={max_len}")

        # Generate general rules
        all_transactions = [tx['products'] for tx in self.transaction_contexts]

        # Sample transactions if too large
        if len(all_transactions) > 20000:
            print("Sampling transactions for rule generation...")
            sample_size = min(20000, len(all_transactions))
            indices = np.random.choice(len(all_transactions), sample_size, replace=False)
            all_transactions = [all_transactions[i] for i in indices]

        general_rules = self._generate_rules_optimized(all_transactions, min_support, min_confidence, max_len)

        if not general_rules.empty:
            self.contextual_rules['general'] = general_rules
            print(f"Generated {len(general_rules)} general rules")

        # Generate context rules for larger datasets
        if len(self.transaction_contexts) >= 1000:
            self._generate_context_rules_optimized(min_support * 1.5, min_confidence, max_len)

    def _generate_rules_optimized(self, transactions_list, min_support, min_confidence, max_len=2):
        """Optimized rule generation with performance tweaks"""
        if len(transactions_list) < 10:
            return pd.DataFrame()

        try:
            # Clean and filter transactions
            cleaned_transactions = []
            product_frequency = {}

            for transaction in transactions_list:
                cleaned_transaction = [str(int(item)) for item in transaction if pd.notna(item)]
                if cleaned_transaction:
                    cleaned_transactions.append(cleaned_transaction)
                    for item in cleaned_transaction:
                        product_frequency[item] = product_frequency.get(item, 0) + 1

            if len(cleaned_transactions) < 10:
                return pd.DataFrame()

            # Filter out rare products to reduce dimensionality
            min_freq = len(cleaned_transactions) * min_support
            frequent_products = {pid for pid, freq in product_frequency.items() if freq >= min_freq}

            filtered_transactions = []
            for transaction in cleaned_transactions:
                filtered_tx = [item for item in transaction if item in frequent_products]
                if len(filtered_tx) >= 1:  # Keep transactions with at least 1 frequent item
                    filtered_transactions.append(filtered_tx)

            print(f"Filtered to {len(filtered_transactions)} transactions with {len(frequent_products)} frequent products")

            if len(filtered_transactions) < 10:
                return pd.DataFrame()

            # Encode transactions
            te = TransactionEncoder()
            te_ary = te.fit(filtered_transactions).transform(filtered_transactions)
            encoded_df = pd.DataFrame(te_ary, columns=te.columns_)

            # Find frequent itemsets with optimized parameters
            frequent_itemsets = apriori(
                encoded_df,
                min_support=min_support,
                use_colnames=True,
                low_memory=True,
                max_len=max_len
            )

            if frequent_itemsets.empty:
                return pd.DataFrame()

            # Generate association rules
            rules = association_rules(
                frequent_itemsets,
                metric="confidence",
                min_threshold=min_confidence
            )

            if not rules.empty:
                rules = rules[rules['lift'] > 1.0]
                rules = rules.sort_values(['confidence', 'support'], ascending=False)

            return rules

        except Exception as e:
            print(f"Error generating rules: {e}")
            return pd.DataFrame()

    def _generate_context_rules_optimized(self, min_support, min_confidence, max_len):
        """Generate context-specific rules efficiently"""
        contexts = [
            ('time_of_day', ['morning', 'afternoon', 'evening', 'night']),
            ('basket_size', ['small', 'medium', 'large']),
        ]

        for context_type, values in contexts:
            for value in values:
                context_key = f"{context_type}_{value}"
                context_transactions = [
                    tx['products'] for tx in self.transaction_contexts
                    if tx[context_type] == value
                ]

                if len(context_transactions) >= 100:  # Higher threshold for context rules
                    context_rules = self._generate_rules_optimized(
                        context_transactions, min_support, min_confidence, max_len
                    )

                    if not context_rules.empty:
                        self.contextual_rules[context_key] = context_rules
                        print(f"Generated {len(context_rules)} rules for {context_key}")

    def _get_day_name(self, dow):
        """Convert day of week number to name"""
        days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        return days[int(dow)] if 0 <= int(dow) < 7 else 'Unknown'

    def _categorize_time_of_day(self, hour):
        """Categorize hour into time segments"""
        if 5 <= hour < 12: return 'morning'
        elif 12 <= hour < 17: return 'afternoon'
        elif 17 <= hour < 22: return 'evening'
        else: return 'night'

    def _categorize_basket_size(self, item_count):
        """Categorize basket size"""
        if item_count <= 5: return 'small'
        elif item_count <= 15: return 'medium'
        else: return 'large'

    def get_recommendations_optimized(self, product_id, context=None, top_n=5):
        """
        Get optimized recommendations
        """
        if context is None:
            context = {'time_of_day': 'afternoon', 'basket_size': 'medium'}

        recommendations = []
        seen_products = set()
        input_product_str = str(product_id)

        # Build context keys
        context_keys = []
        for key, value in context.items():
            context_keys.append(f"{key}_{value}")

        # Search context-specific rules
        for context_key in context_keys:
            if context_key in self.contextual_rules:
                rules = self.contextual_rules[context_key]
                recommendations.extend(self._extract_recommendations(rules, product_id, context_key, seen_products))

        # Fallback to general rules
        if not recommendations and 'general' in self.contextual_rules:
            rules = self.contextual_rules['general']
            recommendations.extend(self._extract_recommendations(rules, product_id, 'general', seen_products))

        # Final fallback
        if not recommendations:
            recommendations = self._get_popular_fallback(product_id, top_n, seen_products)

        recommendations.sort(key=lambda x: x.get('confidence', 0), reverse=True)
        return recommendations[:top_n]

    def _extract_recommendations(self, rules, product_id, context_key, seen_products):
        """Extract recommendations from rules"""
        recommendations = []
        input_product_str = str(product_id)

        for _, rule in rules.iterrows():
            antecedents = [str(item) for item in rule['antecedents']]
            if input_product_str in antecedents:
                consequents = [str(item) for item in rule['consequents']]
                for consequent_id in consequents:
                    if consequent_id != input_product_str and consequent_id not in seen_products:
                        consequent_id_int = int(consequent_id)
                        product_name = self._get_product_name(consequent_id_int)
                        recommendations.append({
                            'product_id': consequent_id_int,
                            'product_name': product_name,
                            'confidence': rule['confidence'],
                            'lift': rule['lift'],
                            'context': context_key,
                            'explanation': f"Frequently bought together in {context_key} (Confidence: {rule['confidence']:.1%})"
                        })
                        seen_products.add(consequent_id)

        return recommendations

    def _get_popular_fallback(self, exclude_product_id, top_n, seen_products):
        """Fallback to popular products"""
        if not self.transaction_contexts:
            return []

        product_counts = {}
        for tx in self.transaction_contexts:
            for product_id in tx['products']:
                if product_id != exclude_product_id and str(product_id) not in seen_products:
                    product_counts[product_id] = product_counts.get(product_id, 0) + 1

        popular_products = sorted(product_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]

        recommendations = []
        for product_id, count in popular_products:
            recommendations.append({
                'product_id': product_id,
                'product_name': self._get_product_name(product_id),
                'confidence': 0.1,
                'lift': 1.0,
                'context': 'popular_fallback',
                'explanation': f"Popular product ({count} transactions)"
            })

        return recommendations

    def _get_product_name(self, product_id):
        """Get product name efficiently"""
        if self.products_df is not None:
            match = self.products_df[self.products_df['product_id'] == product_id]
            if not match.empty:
                return match['product_name'].iloc[0]
        return f"Product_{product_id}"

    def generate_basic_insights(self):
        """
        Generate basic insights without heavy computation
        """
        print("Generating basic insights...")

        insights = {
            'data_summary': {
                'total_orders': self.orders_df['order_id'].nunique(),
                'total_products': self.products_df['product_id'].nunique(),
                'total_customers': self.orders_df['user_id'].nunique(),
                'total_transactions_processed': len(self.transaction_contexts)
            },
            'rule_summary': {
                'total_rules': sum(len(rules) for rules in self.contextual_rules.values()),
                'contexts_available': list(self.contextual_rules.keys()),
                'recommendation_ready': len(self.contextual_rules) > 0
            }
        }

        self.analytics_data = insights
        return insights

    def print_optimization_report(self):
        """Print optimization performance report"""
        print("\n" + "="*60)
        print("OPTIMIZATION PERFORMANCE REPORT")
        print("="*60)

        insights = self.generate_basic_insights()

        print("\nDATA PROCESSING:")
        for key, value in insights['data_summary'].items():
            print(f"  {key.replace('_', ' ').title()}: {value:,}")

        print("\nRULE GENERATION:")
        rule_summary = insights['rule_summary']
        print(f"  Total Rules: {rule_summary['total_rules']}")
        print(f"  Contexts Available: {len(rule_summary['contexts_available'])}")
        print(f"  Recommendation System: {'READY' if rule_summary['recommendation_ready'] else 'NEEDS MORE DATA'}")

        print(f"\nOPTIMIZATION SETTINGS:")
        for key, value in self.optimization_settings.items():
            print(f"  {key}: {value}")

In [ ]:
def main_optimized():
    analytics = GPUOptimizedRetailAnalytics()
    analytics.set_optimization_settings(
        max_transactions=50000,
        sampling_enabled=True,
        sample_size=15000,
        min_support=0.01,
        min_confidence=0.35,
        max_len=2,
        chunk_size=20000
    )
    data_path = '/content/drive/MyDrive/test/dataset'
    sales_data = analytics.load_instacart_data_optimized(data_path)
    analytics.prepare_transactions_memory_efficient()
    analytics.generate_rules_optimized()
    analytics.print_optimization_report()
    if analytics.products_df is not None and not analytics.products_df.empty:
        sample_product_id = analytics.products_df['product_id'].iloc[0]
        product_name = analytics._get_product_name(sample_product_id)

        print(f"\nSAMPLE RECOMMENDATIONS FOR: {product_name}")

        recommendations = analytics.get_recommendations_optimized(
            product_id=sample_product_id,
            context={'time_of_day': 'afternoon', 'basket_size': 'medium'},
            top_n=3
        )

        if recommendations:
            for i, rec in enumerate(recommendations, 1):
                print(f"  {i}. {rec['product_name']} (Confidence: {rec['confidence']:.1%})")
        else:
            print("  No recommendations available")

if __name__ == "__main__":
    main_optimized()

Modified dataset

In [ ]:

class SmartCartSystem:
    def __init__(self):
        self.contextual_rules = {}
        self.products_df = None
        self.transaction_contexts = []
        self.budget_thresholds = {'low': 25, 'medium': 75}

        # Enhanced contextual features per proposal
        self.age_groups = ['teen', 'young_adult', 'adult', 'senior']
        self.genders = ['male', 'female', 'other']
        self.occasions = ['back_to_school', 'holiday', 'weekend', 'weekday', 'special_event']

    def load_and_validate_data(self, csv_file_path):
        """Enhanced data loading with demographic support"""
        print(f"Loading data from {csv_file_path}...")

        try:
            sales_data = pd.read_csv(csv_file_path)
            print(f"Successfully loaded {len(sales_data)} records")

            # Validate required columns from proposal
            required_cols = ['transaction_id', 'product_name']
            missing_cols = [col for col in required_cols if col not in sales_data.columns]

            if missing_cols:
                print(f"Missing required columns: {missing_cols}")
                return None

            # Enhanced validation for demographic data
            demographic_cols = ['customer_age', 'customer_gender', 'occasion']
            available_demographics = [col for col in demographic_cols if col in sales_data.columns]
            print(f"Available demographic features: {available_demographics}")

            # Create product mapping
            if 'product_name' in sales_data.columns:
                product_mapping = sales_data[['product_name']].drop_duplicates()
                product_mapping['product_id'] = range(1, len(product_mapping) + 1)
                self.products_df = product_mapping
                sales_data = sales_data.merge(product_mapping, on='product_name', how='left')

            return self._clean_dataframe(sales_data, 'sales_data')

        except Exception as e:
            print(f"Error loading data: {e}")
            return None

    def _clean_dataframe(self, df, df_name):
        """Enhanced data cleaning"""
        print(f"Cleaning {df_name}...")
        df_clean = df.copy()

        # Handle numeric columns
        numeric_cols = ['quantity', 'unit_price', 'total_item_price', 'total_spent', 'customer_age']
        for col in numeric_cols:
            if col in df_clean.columns:
                if df_clean[col].isnull().sum() > 0:
                    df_clean[col] = df_clean[col].fillna(0)

        # Handle categorical columns
        categorical_cols = ['product_name', 'category', 'time_of_day', 'customer_gender', 'occasion']
        for col in categorical_cols:
            if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
                df_clean[col] = df_clean[col].fillna('unknown')

        # Clean string columns
        string_cols = df_clean.select_dtypes(include=['object']).columns
        for col in string_cols:
            df_clean[col] = df_clean[col].astype(str).str.strip().replace(['', 'nan', 'None'], 'unknown')

        return df_clean

    def prepare_transactions_with_enhanced_demographics(self, sales_df):
        """Enhanced context preparation with demographics as per proposal"""
        print("Preparing transactions with enhanced demographics...")

        # Group products by transaction
        transaction_products = sales_df.groupby('transaction_id')['product_name'].apply(list).reset_index()

        # Calculate transaction statistics
        transaction_stats = sales_df.groupby('transaction_id').agg({
            'product_name': 'count',
            'total_spent': 'first',
            'quantity': 'sum'
        }).reset_index()
        transaction_stats.rename(columns={'product_name': 'item_count'}, inplace=True)

        # Enhanced demographic context
        demographic_info = sales_df.groupby('transaction_id').agg({
            'time_of_day': 'first',
            'date': 'first',
            'customer_age': 'first' if 'customer_age' in sales_df.columns else None,
            'customer_gender': 'first' if 'customer_gender' in sales_df.columns else None,
            'occasion': 'first' if 'occasion' in sales_df.columns else None
        }).reset_index()

        # Process date and age groups
        demographic_info['date'] = pd.to_datetime(demographic_info['date'], errors='coerce')
        demographic_info['day_of_week'] = demographic_info['date'].dt.day_name().fillna('Unknown')

        # Enhanced age categorization
        if 'customer_age' in demographic_info.columns:
            demographic_info['age_group'] = demographic_info['customer_age'].apply(self._categorize_age)
        else:
            demographic_info['age_group'] = 'adult'  # Default

        # Fill missing demographic data
        for col in ['customer_gender', 'occasion']:
            if col in demographic_info.columns:
                demographic_info[col] = demographic_info[col].fillna('unknown')
            else:
                demographic_info[col] = 'unknown'

        # Merge everything
        context_df = transaction_products.merge(transaction_stats, on='transaction_id')
        context_df = context_df.merge(demographic_info, on='transaction_id')

        # Apply categorizations
        context_df['budget_segment'] = context_df['total_spent'].apply(self._categorize_budget)
        context_df['basket_size'] = context_df['item_count'].apply(self._categorize_basket_size)

        # Convert to transaction contexts
        self.transaction_contexts = []
        for _, row in context_df.iterrows():
            if isinstance(row['product_name'], list) and len(row['product_name']) > 0:
                product_ids = []
                for product_name in row['product_name']:
                    product_id = self._get_product_id(product_name)
                    if product_id:
                        product_ids.append(product_id)

                if product_ids:
                    self.transaction_contexts.append({
                        'order_id': row['transaction_id'],
                        'products': product_ids,
                        'budget': row['budget_segment'],
                        'time_of_day': row['time_of_day'],
                        'day_of_week': row['day_of_week'],
                        'basket_size': row['basket_size'],
                        'total_spent': row['total_spent'],
                        'item_count': row['item_count'],
                        # Enhanced demographics
                        'age_group': row['age_group'],
                        'gender': row['customer_gender'],
                        'occasion': row['occasion']
                    })

        print(f"Prepared {len(self.transaction_contexts)} transactions with enhanced demographics")
        self._print_transaction_statistics()
        return self.transaction_contexts

    def _categorize_age(self, age):
        """Categorize age into groups as per proposal"""
        if pd.isna(age):
            return 'adult'
        try:
            age = float(age)
            if age < 18: return 'teen'
            elif age < 25: return 'young_adult'
            elif age < 60: return 'adult'
            else: return 'senior'
        except:
            return 'adult'

    def _categorize_budget(self, total_amount):
        """Categorize total order amount into budget segments"""
        if total_amount <= self.budget_thresholds['low']:
            return 'low'
        elif total_amount <= self.budget_thresholds['medium']:
            return 'medium'
        else:
            return 'high'

    def _categorize_basket_size(self, item_count):
        """Categorize basket size"""
        if item_count <= 3: return 'small'
        elif item_count <= 8: return 'medium'
        else: return 'large'

    def _get_product_id(self, product_name):
        """Get product ID from product name"""
        if self.products_df is not None and not self.products_df.empty:
            match = self.products_df[self.products_df['product_name'] == product_name]
            if not match.empty:
                return match['product_id'].iloc[0]
        return None

    def _get_product_name(self, product_id):
        """Get product name from ID with fallback"""
        if self.products_df is not None and not self.products_df.empty:
            match = self.products_df[self.products_df['product_id'] == product_id]
            if not match.empty:
                return match['product_name'].iloc[0]
        return f"Product_{product_id}"

    def _print_transaction_statistics(self):
        """Print detailed transaction statistics"""
        if not self.transaction_contexts:
            return

        total_products = sum(len(tx['products']) for tx in self.transaction_contexts)
        avg_products = total_products / len(self.transaction_contexts)

        print(f"\nTransaction Statistics:")
        print(f"   Total transactions: {len(self.transaction_contexts)}")
        print(f"   Total products: {total_products}")
        print(f"   Average products per transaction: {avg_products:.1f}")

        # Enhanced demographic statistics
        age_dist = {}
        gender_dist = {}
        occasion_dist = {}
        budget_dist = {}
        time_dist = {}

        for tx in self.transaction_contexts:
            age_dist[tx.get('age_group', 'unknown')] = age_dist.get(tx.get('age_group', 'unknown'), 0) + 1
            gender_dist[tx.get('gender', 'unknown')] = gender_dist.get(tx.get('gender', 'unknown'), 0) + 1
            occasion_dist[tx.get('occasion', 'unknown')] = occasion_dist.get(tx.get('occasion', 'unknown'), 0) + 1
            budget_dist[tx.get('budget', 'unknown')] = budget_dist.get(tx.get('budget', 'unknown'), 0) + 1
            time_dist[tx.get('time_of_day', 'unknown')] = time_dist.get(tx.get('time_of_day', 'unknown'), 0) + 1

        print(f"   Age groups: {age_dist}")
        print(f"   Genders: {gender_dist}")
        print(f"   Occasions: {occasion_dist}")
        print(f"   Budget segments: {budget_dist}")
        print(f"   Time of day: {time_dist}")

    def generate_smart_rules(self, adaptive_thresholds=True):
        """Generate association rules with adaptive thresholds"""
        print("Generating association rules...")

        if len(self.transaction_contexts) < 5:
            print("Too few transactions for meaningful rules")
            return

        # Adaptive thresholds based on data size
        if adaptive_thresholds:
            data_size = len(self.transaction_contexts)
            if data_size < 20:
                min_support = 0.1
                min_confidence = 0.5
            elif data_size < 100:
                min_support = 0.05
                min_confidence = 0.4
            else:
                min_support = 0.01
                min_confidence = 0.3
        else:
            min_support = 0.01
            min_confidence = 0.3

        print(f"   Using adaptive thresholds: support={min_support}, confidence={min_confidence}")

        # Generate general rules
        all_transactions = [tx['products'] for tx in self.transaction_contexts]
        general_rules = self._generate_rules_with_fallback(all_transactions, min_support, min_confidence)

        if not general_rules.empty:
            self.contextual_rules['general'] = general_rules
            print(f"Generated {len(general_rules)} general rules")

            # Show top rules
            if not general_rules.empty:
                print("   Top rules:")
                for i, (_, rule) in enumerate(general_rules.head(3).iterrows(), 1):
                    antecedents = [self._get_product_name(int(x)) for x in rule['antecedents']][:2]
                    consequents = [self._get_product_name(int(x)) for x in rule['consequents']][:2]
                    print(f"     {i}. {antecedents} -> {consequents} (conf: {rule['confidence']:.1%})")
        else:
            print("No general rules generated - data may be too sparse")

        # Generate demographic context rules
        self._generate_demographic_context_rules(min_support * 1.5, min_confidence)

    def _generate_rules_with_fallback(self, transactions_list, min_support, min_confidence):
        """Generate rules with fallback for small datasets"""
        if len(transactions_list) < 3:
            return pd.DataFrame()

        try:
            # Clean transactions
            cleaned_transactions = []
            for transaction in transactions_list:
                cleaned_transaction = [str(int(item)) for item in transaction if pd.notna(item)]
                if cleaned_transaction:
                    cleaned_transactions.append(cleaned_transaction)

            if len(cleaned_transactions) < 3:
                return pd.DataFrame()

            # For very small datasets, use more lenient parameters
            if len(cleaned_transactions) < 10:
                min_support = max(0.05, min_support)
                max_len = 2
            else:
                max_len = 3

            # Encode transactions
            te = TransactionEncoder()
            te_ary = te.fit(cleaned_transactions).transform(cleaned_transactions)
            encoded_df = pd.DataFrame(te_ary, columns=te.columns_)

            # Find frequent itemsets
            frequent_itemsets = apriori(
                encoded_df,
                min_support=min_support,
                use_colnames=True,
                low_memory=True,
                max_len=max_len
            )

            if frequent_itemsets.empty:
                return pd.DataFrame()

            # Generate association rules
            rules = association_rules(
                frequent_itemsets,
                metric="confidence",
                min_threshold=min_confidence
            )

            if not rules.empty:
                rules = rules[rules['lift'] > 1.0]  # Positive correlation
                rules = rules.sort_values(['confidence', 'support'], ascending=False)

            return rules

        except Exception as e:
            print(f"Error generating rules: {e}")
            return pd.DataFrame()

    def _generate_demographic_context_rules(self, min_support, min_confidence):
        """Generate demographic context-specific rules"""
        contexts = [
            ('age_group', ['teen', 'young_adult', 'adult', 'senior']),
            ('gender', ['male', 'female']),
            ('time_of_day', ['morning', 'afternoon', 'evening']),
            ('budget', ['low', 'medium', 'high']),
            ('occasion', ['back_to_school', 'holiday', 'weekend'])
        ]

        for context_type, values in contexts:
            for value in values:
                context_key = f"{context_type}_{value}"
                context_transactions = [
                    tx['products'] for tx in self.transaction_contexts
                    if tx.get(context_type) == value
                ]

                if len(context_transactions) >= 3:  # Lower threshold for demographic contexts
                    context_rules = self._generate_rules_with_fallback(
                        context_transactions, min_support, min_confidence
                    )

                    if not context_rules.empty:
                        self.contextual_rules[context_key] = context_rules
                        print(f"Generated {len(context_rules)} rules for {context_key}")

    def generate_occasion_based_recommendations(self, occasion: str, top_n: int = 5):
        """Generate occasion-based bundles as per proposal"""
        print(f"Generating {occasion}-based recommendations...")

        # Filter transactions by occasion
        occasion_transactions = [
            tx for tx in self.transaction_contexts
            if tx.get('occasion') == occasion
        ]

        if not occasion_transactions:
            print(f"No transactions found for occasion: {occasion}")
            return self._get_general_popular_products(top_n)

        # Generate rules for this occasion
        transactions_list = [tx['products'] for tx in occasion_transactions]
        occasion_rules = self._generate_rules_with_fallback(transactions_list, 0.05, 0.3)

        if occasion_rules.empty:
            return self._get_occasion_fallback_products(occasion, top_n)

        # Extract top recommendations
        recommendations = []
        seen_products = set()

        for _, rule in occasion_rules.iterrows():
            if len(rule['consequents']) > 0:
                for product_id in rule['consequents']:
                    product_id_int = int(product_id)
                    if product_id_int not in seen_products:
                        product_name = self._get_product_name(product_id_int)
                        recommendations.append({
                            'product_id': product_id_int,
                            'product_name': product_name,
                            'confidence': rule['confidence'],
                            'context': f'occasion_{occasion}',
                            'explanation': f"Frequently purchased during {occasion.replace('_', ' ')} (confidence: {rule['confidence']:.1%})"
                        })
                        seen_products.add(product_id_int)

                        if len(recommendations) >= top_n * 2:
                            break

        recommendations.sort(key=lambda x: x['confidence'], reverse=True)
        return recommendations[:top_n]

    def _get_occasion_fallback_products(self, occasion: str, top_n: int):
        """Fallback products for occasions"""
        # Define occasion-specific fallback products
        occasion_fallbacks = {
            'back_to_school': ['Notebooks', 'Pens', 'Backpack', 'Calculator', 'Textbooks'],
            'holiday': ['Chocolates', 'Wine', 'Gift Basket', 'Decorations', 'Cookies'],
            'weekend': ['Snacks', 'Drinks', 'Movie', 'Games', 'BBQ Supplies']
        }

        fallbacks = occasion_fallbacks.get(occasion, ['Popular Item 1', 'Popular Item 2', 'Popular Item 3'])
        recommendations = []

        for product_name in fallbacks[:top_n]:
            product_id = self._get_product_id(product_name)
            if product_id:
                recommendations.append({
                    'product_id': product_id,
                    'product_name': product_name,
                    'confidence': 0.1,
                    'context': f'occasion_{occasion}_fallback',
                    'explanation': f"Commonly purchased for {occasion.replace('_', ' ')} events"
                })

        return recommendations

    def _get_general_popular_products(self, top_n: int):
        """Get generally popular products as fallback"""
        if not self.transaction_contexts:
            return []

        product_counts = {}
        for tx in self.transaction_contexts:
            for product_id in tx['products']:
                product_counts[product_id] = product_counts.get(product_id, 0) + 1

        popular_products = sorted(product_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]

        recommendations = []
        for product_id, count in popular_products:
            recommendations.append({
                'product_id': product_id,
                'product_name': self._get_product_name(product_id),
                'confidence': 0.1,
                'context': 'popular_fallback',
                'explanation': f"Popular product (appears in {count} transactions)"
            })

        return recommendations

    def get_demographic_recommendations(self, product_name: str, age_group: str = None,
                                      gender: str = None, budget: str = 'medium',
                                      time_of_day: str = 'afternoon', top_n: int = 5):
        """Get recommendations based on demographic profile as per proposal"""
        # Convert product name to ID
        product_id = self._get_product_id(product_name)
        if product_id is None:
            print(f"Product '{product_name}' not found in database")
            return []

        context = {
            'age_group': age_group,
            'gender': gender,
            'budget': budget,
            'time_of_day': time_of_day
        }

        # Filter context keys that are provided
        context_keys = []
        for key, value in context.items():
            if value is not None:
                context_keys.append(f"{key}_{value}")

        recommendations = []
        seen_products = set()

        # Try demographic-specific rules first
        for context_key in context_keys:
            if context_key in self.contextual_rules:
                rules = self.contextual_rules[context_key]
                input_product_str = str(product_id)

                for _, rule in rules.iterrows():
                    antecedents = [str(item) for item in rule['antecedents']]
                    if input_product_str in antecedents:
                        consequents = [str(item) for item in rule['consequents']]
                        for consequent_id in consequents:
                            if consequent_id != input_product_str and consequent_id not in seen_products:
                                consequent_id_int = int(consequent_id)
                                rec_product_name = self._get_product_name(consequent_id_int)
                                recommendations.append({
                                    'product_id': consequent_id_int,
                                    'product_name': rec_product_name,
                                    'confidence': rule['confidence'],
                                    'context': context_key,
                                    'explanation': f"Frequently bought together by {context_key.replace('_', ' ')} customers (confidence: {rule['confidence']:.1%})"
                                })
                                seen_products.add(consequent_id)

        # Fallback to general rules
        if not recommendations and 'general' in self.contextual_rules:
            rules = self.contextual_rules['general']
            input_product_str = str(product_id)

            for _, rule in rules.iterrows():
                antecedents = [str(item) for item in rule['antecedents']]
                if input_product_str in antecedents:
                    consequents = [str(item) for item in rule['consequents']]
                    for consequent_id in consequents:
                        if consequent_id != input_product_str and consequent_id not in seen_products:
                            consequent_id_int = int(consequent_id)
                            rec_product_name = self._get_product_name(consequent_id_int)
                            recommendations.append({
                                'product_id': consequent_id_int,
                                'product_name': rec_product_name,
                                'confidence': rule['confidence'],
                                'context': 'general',
                                'explanation': f"Frequently bought together (confidence: {rule['confidence']:.1%})"
                            })
                            seen_products.add(consequent_id)

        # Final fallback
        if not recommendations:
            fallback_recs = self._get_general_popular_products(top_n)
            # Remove the input product from fallback if present
            recommendations = [rec for rec in fallback_recs if rec['product_id'] != product_id]

        # Sort and return
        recommendations.sort(key=lambda x: x.get('confidence', 0), reverse=True)
        return recommendations[:top_n]

    def generate_comprehensive_report(self):
        """Enhanced report matching proposal requirements"""
        print("\n" + "="*60)
        print("SMARTCART COMPREHENSIVE ANALYTICS REPORT")
        print("="*60)

        report = {
            'system_overview': {
                'system_name': 'SmartCart',
                'version': '1.0',
                'key_features': [
                    'Market Basket Analysis with Apriori/FP-Growth',
                    'Demographic-aware recommendations (age, gender)',
                    'Occasion-based product bundles',
                    'Contextual personalization'
                ]
            },
            'data_analysis': self._get_data_summary(),
            'demographic_coverage': self._get_demographic_coverage(),
            'recommendation_quality': self._get_recommendation_quality(),
            'business_impact': self._get_business_impact_estimation()
        }

        self._print_enhanced_report(report)
        return report

    def _get_data_summary(self):
        """Generate data summary"""
        if not self.transaction_contexts:
            return {"message": "No data available"}

        total_products = sum(len(tx['products']) for tx in self.transaction_contexts)
        unique_products = len(set(product for tx in self.transaction_contexts for product in tx['products']))

        return {
            'total_transactions': len(self.transaction_contexts),
            'total_products_in_transactions': total_products,
            'unique_products': unique_products,
            'average_products_per_transaction': total_products / len(self.transaction_contexts),
            'data_sufficiency': 'Good' if len(self.transaction_contexts) >= 50 else 'Limited'
        }

    def _get_demographic_coverage(self):
        """Analyze demographic data coverage"""
        if not self.transaction_contexts:
            return {"message": "No demographic data available"}

        age_dist = {}
        gender_dist = {}
        occasion_dist = {}

        for tx in self.transaction_contexts:
            age_dist[tx.get('age_group', 'unknown')] = age_dist.get(tx.get('age_group', 'unknown'), 0) + 1
            gender_dist[tx.get('gender', 'unknown')] = gender_dist.get(tx.get('gender', 'unknown'), 0) + 1
            occasion_dist[tx.get('occasion', 'unknown')] = occasion_dist.get(tx.get('occasion', 'unknown'), 0) + 1

        return {
            'age_group_distribution': age_dist,
            'gender_distribution': gender_dist,
            'occasion_distribution': occasion_dist,
            'demographic_coverage': 'Good' if len(age_dist) > 1 else 'Limited'
        }

    def _get_recommendation_quality(self):
        """Assess recommendation system quality"""
        total_rules = sum(len(rules) for rules in self.contextual_rules.values())

        if total_rules >= 20:
            status = 'Excellent'
        elif total_rules >= 10:
            status = 'Good'
        elif total_rules >= 5:
            status = 'Limited'
        else:
            status = 'Poor'

        return {
            'status': status,
            'total_rules': total_rules,
            'contexts_available': list(self.contextual_rules.keys()),
            'message': f'System has {total_rules} rules across {len(self.contextual_rules)} contexts'
        }

    def _get_business_impact_estimation(self):
        """Estimate potential business impact"""
        if not self.transaction_contexts:
            return {"message": "Insufficient data for impact estimation"}

        avg_basket_size = sum(tx['item_count'] for tx in self.transaction_contexts) / len(self.transaction_contexts)
        avg_spend = sum(tx['total_spent'] for tx in self.transaction_contexts) / len(self.transaction_contexts)

        # Simple impact estimation
        potential_upsell = avg_spend * 0.15  # 15% potential increase
        potential_cross_sell = avg_basket_size * 0.20  # 20% potential increase in basket size

        return {
            'average_basket_size': round(avg_basket_size, 2),
            'average_spend': round(avg_spend, 2),
            'potential_upsell_increase': round(potential_upsell, 2),
            'potential_cross_sell_increase': round(potential_cross_sell, 2),
            'estimated_impact': 'High' if potential_upsell > 10 else 'Moderate'
        }

    def _print_enhanced_report(self, report):
        """Print the comprehensive report"""
        # System Overview
        print("\nSYSTEM OVERVIEW:")
        overview = report['system_overview']
        print(f"   Name: {overview['system_name']}")
        print(f"   Version: {overview['version']}")
        print("   Key Features:")
        for feature in overview['key_features']:
            print(f"     - {feature}")

        # Data Analysis
        print("\nDATA ANALYSIS:")
        for key, value in report['data_analysis'].items():
            print(f"   {key.replace('_', ' ').title()}: {value}")

        # Demographic Coverage
        print("\nDEMOGRAPHIC COVERAGE:")
        demo_coverage = report['demographic_coverage']
        for key, value in demo_coverage.items():
            if key != 'demographic_coverage':
                print(f"   {key.replace('_', ' ').title()}: {value}")
        print(f"   Overall Coverage: {demo_coverage['demographic_coverage']}")

        # Recommendation Quality
        print("\nRECOMMENDATION QUALITY:")
        quality = report['recommendation_quality']
        print(f"   Status: {quality['status']}")
        print(f"   {quality['message']}")

        # Business Impact
        print("\nBUSINESS IMPACT ESTIMATION:")
        impact = report['business_impact']
        for key, value in impact.items():
            if key != 'estimated_impact':
                print(f"   {key.replace('_', ' ').title()}: {value}")
        print(f"   Estimated Impact: {impact['estimated_impact']}")



In [ ]:

def main():
    smartcart = SmartCartSystem()

    # Load data
    sales_data = smartcart.load_and_validate_data('/content/smartcart_transactions2.csv')

    if sales_data is not None:
        # Prepare transactions with demographics
        smartcart.prepare_transactions_with_enhanced_demographics(sales_data)

        # Generate rules
        smartcart.generate_smart_rules(adaptive_thresholds=True)

        # Test occasion-based recommendations
        print("\n" + "="*50)
        print("OCCASION-BASED RECOMMENDATIONS")
        print("="*50)

        occasions = ['back_to_school', 'holiday', 'weekend']
        for occasion in occasions:
            occasion_recs = smartcart.generate_occasion_based_recommendations(occasion, top_n=3)
            print(f"\n{occasion.replace('_', ' ').title()} Recommendations:")
            if occasion_recs:
                for i, rec in enumerate(occasion_recs, 1):
                    print(f"  {i}. {rec['product_name']} - {rec['explanation']}")
            else:
                print("  No specific recommendations available")

        # Test demographic recommendations
        print("\n" + "="*50)
        print("DEMOGRAPHIC RECOMMENDATIONS")
        print("="*50)

        # Test with a sample product
        sample_products = ['Chips', 'Water Bottles', 'Notebooks']
        for product in sample_products[:1]:  # Test with first product
            print(f"\nRecommendations for customers buying '{product}':")

            # Young adult female
            demo_recs = smartcart.get_demographic_recommendations(
                product_name=product,
                age_group='young_adult',
                gender='female',
                budget='medium',
                top_n=3
            )

            print(f"  Young Adult Female (Medium Budget):")
            if demo_recs:
                for i, rec in enumerate(demo_recs, 1):
                    print(f"    {i}. {rec['product_name']} - {rec['explanation']}")
            else:
                print("    No specific recommendations available")

        # Generate comprehensive report
        print("\n" + "="*50)
        smartcart.generate_comprehensive_report()

if __name__ == "__main__":
    main()